In [4]:
# Cell 1: Install + clone repos
!pip install -q transformers datasets accelerate
import os, sys

# Clone ROME
if not os.path.exists("/content/rome"):
    !git clone https://github.com/kmeng01/rome.git /content/rome
    print("ROME cloned")
else:
    print("ROME already present")

# Clone MEMIT
if not os.path.exists("/content/memit"):
    !git clone https://github.com/kmeng01/memit.git /content/memit
    print("MEMIT cloned")
else:
    print("MEMIT already present")

# Install ROME requirements
os.chdir("/content/rome")
!pip install -q -r requirements.txt

# Add both to path
sys.path.insert(0, "/content/rome")
sys.path.insert(0, "/content/memit")

print("Done")

Cloning into '/content/rome'...
remote: Enumerating objects: 768, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 768 (delta 343), reused 250 (delta 250), pack-reused 329 (from 1)
Receiving objects: 100% (768/768), 22.69 MiB | 14.34 MiB/s, done.
Resolving deltas: 100% (460/460), done.
ROME cloned
Cloning into '/content/memit'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 196 (delta 47), reused 40 (delta 40), pack-reused 99 (from 1)
Receiving objects: 100% (196/196), 134.90 KiB | 12.26 MiB/s, done.
Resolving deltas: 100% (59/59), done.
MEMIT cloned
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Done


In [5]:
# Cell 2: Load GPT-2-XL
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc, json, requests

MODEL_NAME = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# GPT-2-XL config patches needed by ROME/MEMIT
if not hasattr(model.config, 'n_positions'):
    model.config.n_positions = model.config.max_position_embeddings if hasattr(model.config, 'max_position_embeddings') else 1024
if not hasattr(model.config, 'n_embd'):
    model.config.n_embd = model.config.hidden_size

print(f"Loaded {MODEL_NAME}")
print(f"  Layers: {model.config.n_layer}")
print(f"  Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2-xl
  Layers: 48
  Free GPU: 20.3 GB


In [6]:
# Cell 3: Helpers
def get_prob(m, prompt, target_str):
    target_id = tokenizer.encode(" " + target_str.strip(), add_special_tokens=False)[0]
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        logits = m(**inputs).logits[0, -1, :]
    return torch.softmax(logits, dim=-1)[target_id].item()

def generate(m, prompt, max_new=15):
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def safe_mean(lst):
    vals = [x for x in lst if x is not None]
    return sum(vals) / len(vals) if vals else None

cf = requests.get("https://rome.baulab.info/data/dsets/counterfact.json").json()
print(f"CounterFact: {len(cf)} samples")

CounterFact: 21919 samples


In [28]:
# Patch layer_stats.py to use wikitext-103 instead of wikipedia
with open("/content/memit/rome/layer_stats.py", "r") as f:
    src = f.read()

# Fix the dataset lookup dict — wikipedia uses deprecated scripts, wikitext works
src = src.replace(
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en")[ds_name]',
    '"wikitext-103-raw-v1"'  # always use wikitext regardless of ds_name
)

with open("/content/memit/rome/layer_stats.py", "w") as f:
    f.write(src)

# Also reload the module so the change takes effect
import importlib, sys
for mod_name in list(sys.modules.keys()):
    if "layer_stats" in mod_name or "memit_main" in mod_name:
        del sys.modules[mod_name]

from memit.memit_main import MEMITHyperParams, apply_memit_to_model
print("layer_stats.py patched and reloaded")

# Verify the patch landed
with open("/content/memit/rome/layer_stats.py") as f:
    for i, line in enumerate(f):
        if "wikitext" in line and i < 110:
            print(f"  line {i+1}: {line.rstrip()}")

layer_stats.py patched and reloaded
  line 39:     aa("--dataset", default="wikipedia", choices=["wikitext", "wikipedia"])
  line 99:             "wikitext-103-raw-v1",


In [29]:
import json, os

hparams_dir = "/content/memit/hparams/MEMIT"
os.makedirs(hparams_dir, exist_ok=True)
path = f"{hparams_dir}/gpt2-xl.json"

hparams_data = {
    "layers": [13, 14, 15, 16, 17],
    "layer_selection": "all",
    "fact_token": "subject_last",
    "v_num_grad_steps": 20,
    "v_lr": 0.5,
    "v_loss_layer": 47,
    "v_weight_decay": 0.5,
    "clamp_norm_factor": 4,
    "kl_factor": 0.0625,
    "mom2_adjustment": True,
    "mom2_update_weight": 20000,
    "rewrite_module_tmp": "transformer.h.{}.mlp.c_proj",
    "layer_module_tmp": "transformer.h.{}",
    "mlp_module_tmp": "transformer.h.{}.mlp",
    "attn_module_tmp": "transformer.h.{}.attn",
    "ln_f_module": "transformer.ln_f",
    "lm_head_module": "transformer.wte",        # ← fixed
    "mom2_dataset": "wikipedia",     # ← fixed
    "mom2_n_samples": 3000,
    "mom2_dtype": "float32"
}

with open(path, "w") as f:
    json.dump(hparams_data, f, indent=2)
print(f"Written: {path}")

# Reload hparams
from memit.memit_main import MEMITHyperParams, apply_memit_to_model
hparams = MEMITHyperParams.from_json(path)
print("Hparams loaded:", hparams.rewrite_module_tmp)
print("Layers:", hparams.layers)

Written: /content/memit/hparams/MEMIT/gpt2-xl.json
Hparams loaded: transformer.h.{}.mlp.c_proj
Layers: [13, 14, 15, 16, 17]


In [17]:
import yaml, os

# Must be written to /content/memit since globals.py opens "globals.yml" relative to cwd
os.chdir("/content/memit")

config = {
    "RESULTS_DIR":    "/content/memit/results",
    "DATA_DIR":       "/content/memit/data",
    "STATS_DIR":      "/content/memit/data/stats",
    "HPARAMS_DIR":    "/content/memit/hparams",
    "KV_DIR":         "/content/memit/data/kv",
    "REMOTE_ROOT_URL": "https://rome.baulab.info",
}

for k, v in config.items():
    if k != "REMOTE_ROOT_URL":
        os.makedirs(v, exist_ok=True)

with open("/content/memit/globals.yml", "w") as f:
    yaml.dump(config, f)

print("globals.yml written correctly")
print(open("/content/memit/globals.yml").read())

globals.yml written correctly
DATA_DIR: /content/memit/data
HPARAMS_DIR: /content/memit/hparams
KV_DIR: /content/memit/data/kv
REMOTE_ROOT_URL: https://rome.baulab.info
RESULTS_DIR: /content/memit/results
STATS_DIR: /content/memit/data/stats



In [18]:
# Cell 5: Apply MEMIT patches
with open("/content/memit/memit/memit_main.py", "r") as f:
    src = f.read()

patches = [
    (
        "upd_matrix = key_mat @ val_mat.T",
        "upd_matrix = key_mat.float() @ val_mat.float().T"
    ),
    (
        "upd_matrix = resid @ adj_k.T",
        "upd_matrix = resid.float() @ adj_k.float().T"
    ),
]

for old, new in patches:
    if old in src:
        src = src.replace(old, new)
        print(f"Patched: {old[:60]}")
    else:
        print(f"Already patched or not found: {old[:60]}")

with open("/content/memit/memit/memit_main.py", "w") as f:
    f.write(src)

import importlib
sys.path.insert(0, "/content/memit")
import memit.memit_main
importlib.reload(memit.memit_main)
from memit.memit_main import MEMITHyperParams, apply_memit_to_model
print("apply_memit_to_model loaded")

Already patched or not found: upd_matrix = key_mat @ val_mat.T
Already patched or not found: upd_matrix = resid @ adj_k.T
apply_memit_to_model loaded


In [15]:
# Run this to see what globals.py actually reads
with open("/content/memit/util/globals.py") as f:
    print(f.read())

from pathlib import Path

import yaml

with open("globals.yml", "r") as stream:
    data = yaml.safe_load(stream)

(RESULTS_DIR, DATA_DIR, STATS_DIR, HPARAMS_DIR, KV_DIR) = (
    Path(z)
    for z in [
        data["RESULTS_DIR"],
        data["DATA_DIR"],
        data["STATS_DIR"],
        data["HPARAMS_DIR"],
        data["KV_DIR"],
    ]
)

REMOTE_ROOT_URL = data["REMOTE_ROOT_URL"]



In [16]:
with open("/content/memit/globals.yml") as f:
    print(f.read())

DATA_DIR: /content/memit/data
HPARAMS_DIR: /content/memit/hparams
KV_DIR: /content/memit/data/kv
STATS_DIR: /content/memit/data/stats



In [31]:
# ── Nuclear fix for layer_stats.py ───────────────────────────────────────────
# Replaces the entire get_ds() body to use wikitext via the modern datasets API

with open("/content/memit/rome/layer_stats.py", "r") as f:
    src = f.read()

# Replace the load_dataset call that passes ds_name (="wikipedia") directly
old = '''        raw_ds = load_dataset(
            ds_name,
            dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en")[ds_name],
        )'''

new = '''        raw_ds = load_dataset("wikitext", "wikitext-103-raw-v1")'''

if old in src:
    src = src.replace(old, new)
    print("Patched load_dataset call")
else:
    print("String not found — printing the get_ds function so you can see what's there:")
    start = src.find("def get_ds")
    print(src[start:start+400])

with open("/content/memit/rome/layer_stats.py", "w") as f:
    f.write(src)

# Force full reload of every cached module that touches layer_stats
import sys, importlib
to_drop = [k for k in sys.modules if any(x in k for x in
           ["layer_stats", "memit_main", "compute_z", "compute_ks",
            "rome_main", "repr_tools", "nethook"])]
for k in to_drop:
    del sys.modules[k]
print(f"Dropped {len(to_drop)} cached modules: {to_drop}")

# Re-import fresh
from memit.memit_main import MEMITHyperParams, apply_memit_to_model
print("Reloaded cleanly")

# Verify the patch is in the file
with open("/content/memit/rome/layer_stats.py") as f:
    for i, line in enumerate(f, 1):
        if "load_dataset" in line:
            print(f"  line {i}: {line.rstrip()}")

String not found — printing the get_ds function so you can see what's there:
def get_ds():
        raw_ds = load_dataset(
            ds_name,
            "wikitext-103-raw-v1",
        )
        maxlen = model.config.n_positions
        if batch_tokens is not None and batch_tokens < maxlen:
            maxlen = batch_tokens
        return TokenizedDataset(raw_ds["train"], tokenizer, maxlen=maxlen)

    # Continue with computation of statistics
    batch_size = 100  # Exam
Dropped 7 cached modules: ['util.nethook', 'rome.repr_tools', 'rome.rome_main', 'memit.compute_z', 'memit.compute_ks', 'rome.layer_stats', 'memit.memit_main']
Reloaded cleanly
  line 5: from datasets import load_dataset
  line 97:         raw_ds = load_dataset(


In [32]:
with open("/content/memit/rome/layer_stats.py", "r") as f:
    src = f.read()

# Fix: replace ds_name with the literal "wikitext" as the dataset name
old = '''        raw_ds = load_dataset(
            ds_name,
            "wikitext-103-raw-v1",
        )'''

new = '''        raw_ds = load_dataset(
            "wikitext",
            "wikitext-103-raw-v1",
        )'''

if old in src:
    src = src.replace(old, new)
    print("Patched: ds_name → 'wikitext'")
else:
    print("Not found — current load_dataset block:")
    i = src.find("raw_ds = load_dataset")
    print(repr(src[i-8:i+120]))

with open("/content/memit/rome/layer_stats.py", "w") as f:
    f.write(src)

# Drop and reload
import sys
for k in [k for k in sys.modules if any(x in k for x in
          ["layer_stats","memit_main","compute_z","compute_ks","rome_main","repr_tools","nethook"])]:
    del sys.modules[k]

from memit.memit_main import MEMITHyperParams, apply_memit_to_model
print("Done — re-run Cell 6 now")

Patched: ds_name → 'wikitext'
Done — re-run Cell 6 now


In [33]:
# Cell 6: Load hparams + smoke test
hparams = MEMITHyperParams.from_json(path)
print("Hparams loaded:", hparams.rewrite_module_tmp)

# Smoke test on cf[0]
sample = cf[0]
rw = sample["requested_rewrite"]
prompt = rw["prompt"].format(rw["subject"])
target_new = rw["target_new"]["str"]

print(f"Before: '{generate(model, prompt)}'")
print(f"P(new) before: {get_prob(model, prompt, target_new):.4f}")

request = {
    "prompt":     rw["prompt"],
    "subject":    rw["subject"],
    "target_new": {"str": target_new}
}

edited_model, _ = apply_memit_to_model(
    model, tokenizer, [request], hparams,
    copy=True, return_orig_weights=True
)

print(f"After:  '{generate(edited_model, prompt)}'")
print(f"P(new) after: {get_prob(edited_model, prompt, target_new):.4f}")
del edited_model
torch.cuda.empty_cache()

Hparams loaded: transformer.h.{}.mlp.c_proj
Before: 'French. She was born in Montreal, Canada and moved to the United States'
P(new) before: 0.0518
MEMIT request sample: [The mother tongue of Danielle Darrieux is] -> [ English]
Cached context templates [['{}'], ['The last two of the lastInvalid \n. {}', 'Therefore, the only a lot, we Cosponsors are. {}', 'Because I waspsuch, in a lotz. {}', 'I think the way) is a loteInvalid. {}', 'You have to the Reader (probablyluajrawdownloadcloneembedreportprint:. {}']]
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 3.012 = 3.012 + 0.0 + 0.0 avg prob of [ English] 0.05059819668531418
loss 2.118 = 2.116 + 0.001 + 0.001 avg prob of [ English] 0.12909650802612305
loss 1.85 = 1.846 + 0.002 + 0.002 avg prob of [ English] 0.16963708400726318
loss 1.554 = 1.549 + 0.003 + 0.002 avg prob of [ English] 

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

orig norm tensor(113.0625, device='cuda:0', dtype=torch.float16)
upd norm tensor(1.2324, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


LAYER 16

Writing 1 key/value pair(s) into layer 16
z error tensor(86.5239, device='cuda:0', grad_fn=<MeanBackward0>)
Retrieving covariance statistics for gpt2-xl @ transformer.h.16.mlp.c_proj.
Attempting to download gpt2-xl/wikipedia_stats/transformer.h.16.mlp.c_proj_float32_mom2_3000.npz from https://rome.baulab.info/data/stats/gpt2-xl/wikipedia_stats/transformer.h.16.mlp.c_proj_float32_mom2_3000.npz.
Unable to download due to HTTP Error 404: Not Found. Computing locally....


  0%|          | 0/30 [00:00<?, ?it/s]

orig norm tensor(114., device='cuda:0', dtype=torch.float16)
upd norm tensor(1.5256, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


LAYER 17

Writing 1 key/value pair(s) into layer 17
z error tensor(71.2244, device='cuda:0', grad_fn=<MeanBackward0>)
Retrieving covariance statistics for gpt2-xl @ transformer.h.17.mlp.c_proj.
Attempting to download gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_3000.npz from https://rome.baulab.info/data/stats/gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_3000.npz.
Unable to download due to HTTP Error 404: Not Found. Computing locally....


  0%|          | 0/30 [00:00<?, ?it/s]

orig norm tensor(117.1250, device='cuda:0', dtype=torch.float16)
upd norm tensor(2.4173, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Deltas successfully computed for ['transformer.h.13.mlp.c_proj.weight', 'transformer.h.14.mlp.c_proj.weight', 'transformer.h.15.mlp.c_proj.weight', 'transformer.h.16.mlp.c_proj.weight', 'transformer.h.17.mlp.c_proj.weight']
New weights successfully inserted into ['transformer.h.13.mlp.c_proj.weight', 'transformer.h.14.mlp.c_proj.weight', 'transformer.h.15.mlp.c_proj.weight', 'transformer.h.16.mlp.c_proj.weight', 'transformer.h.17.mlp.c_proj.weight']
After:  'English. She was born in London, England, to Richard and Mary ('
P(new) after: 0.9980


In [34]:
# Cell 7: Full 50-sample run
results_memit = []

print("Running MEMIT on 50 CounterFact samples...")
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB\n")

for i, sample in enumerate(cf[:50]):
    rw          = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    request = {
        "prompt":     rw["prompt"],
        "subject":    rw["subject"],
        "target_new": {"str": target_new}
    }

    baseline_p = get_prob(model, prompt, target_new)
    gen_before = generate(model, prompt)

    try:
        edited_model, _ = apply_memit_to_model(
            model, tokenizer, [request], hparams,
            copy=True, return_orig_weights=True
        )

        p_after           = get_prob(edited_model, prompt, target_new)
        gen_after         = generate(edited_model, prompt)
        edit_success      = p_after

        para_prompts = sample.get("paraphrase_prompts", [])
        para_success = safe_mean(
            [get_prob(edited_model, p, target_new) for p in para_prompts[:5]]
        ) if para_prompts else None

        nbr_prompts = sample.get("neighborhood_prompts", [])
        if nbr_prompts:
            bleed_probs, pres_probs = [], []
            base_correct = base_correct_broken = 0
            for p in nbr_prompts[:5]:
                base_p_true = get_prob(model, p, target_true)
                edit_p_new  = get_prob(edited_model, p, target_new)
                edit_p_true = get_prob(edited_model, p, target_true)
                bleed_probs.append(edit_p_new)
                pres_probs.append(edit_p_true)
                if base_p_true > 0.05:
                    base_correct += 1
                    if edit_p_true < base_p_true * 0.5:
                        base_correct_broken += 1
            nbr_bleed        = sum(bleed_probs) / len(bleed_probs)
            nbr_preservation = sum(pres_probs)  / len(pres_probs)
            oe_damage        = base_correct_broken / base_correct if base_correct > 0 else 0.0
        else:
            nbr_bleed = nbr_preservation = oe_damage = None

        del edited_model
        torch.cuda.empty_cache(); gc.collect()

        results_memit.append({
            "idx":                       i,
            "subject":                   rw["subject"],
            "prompt":                    prompt,
            "target_new":                target_new,
            "target_true":               target_true,
            "baseline_p":                baseline_p,
            "edit_prob":                 p_after,
            "edit_success":              edit_success,
            "paraphrase_success":        para_success,
            "over_extinction":           nbr_bleed,
            "neighborhood_preservation": nbr_preservation,
            "oe_damage":                 oe_damage,
            "locality_drop":             None,
            "gen_before":                gen_before,
            "gen_after":                 gen_after,
            "method":                    "MEMIT",
            "model":                     "gpt2-xl",
            "n_steps":                   1,
            "kl_final":                  None,
            "oe_source":                 "live_inference",
            "n_mlp":                     None,
        })

        b = f"{nbr_bleed:.3f}" if nbr_bleed is not None else "N/A"
        p = f"{para_success:.3f}" if para_success is not None else "N/A"
        print(f"[{i:02d}] edit_p={p_after:.3f} bleed={b} para={p} | {gen_after[:45]!r}")

    except Exception as e:
        torch.cuda.empty_cache(); gc.collect()
        print(f"[{i:02d}] ERROR: {e}")
        results_memit.append({"idx": i, "error": str(e), "edit_success": 0.0})

print(f"\nDone. Avg edit_p: {safe_mean([r['edit_success'] for r in results_memit if 'error' not in r]):.3f}")

Running MEMIT on 50 CounterFact samples...
Free GPU: 2.9 GB

MEMIT request sample: [The mother tongue of Danielle Darrieux is] -> [ English]
Computing right vector (v)
Lookup index found: 7 | Sentence: The mother tongue of Danielle Darrieux is | Token: ux
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 3.012 = 3.012 + 0.0 + 0.0 avg prob of [ English] 0.05059819668531418
loss 2.118 = 2.116 + 0.001 + 0.001 avg prob of [ English] 0.12909650802612305
loss 1.85 = 1.846 + 0.002 + 0.002 avg prob of [ English] 0.16963708400726318
loss 1.554 = 1.549 + 0.003 + 0.002 avg prob of [ English] 0.22950857877731323
loss 1.255 = 1.249 + 0.004 + 0.003 avg prob of [ English] 0.31030380725860596
loss 0.947 = 0.939 + 0.005 + 0.003 avg prob of [ English] 0.4226234555244446
loss 0.614 = 0.604 + 0.007 + 0.003 avg prob of [ English] 0.5850213766098022
loss 0.317 = 0.303 + 0.01 + 0.004 avg prob of [ English] 0.7732170820236206
loss 0.141 = 0.122 + 0.016 + 0.004 avg prob 

In [35]:
# Cell 8: Save results
good = [r for r in results_memit if "error" not in r]

summary = {
    "method":    "MEMIT",
    "model":     "gpt2-xl",
    "dataset":   "CounterFact",
    "n_samples": len(good),
    "metrics": {
        "edit_success":              safe_mean([r["edit_success"] for r in good]),
        "paraphrase_success":        safe_mean([r.get("paraphrase_success") for r in good]),
        "neighborhood_bleed":        safe_mean([r.get("over_extinction") for r in good]),
        "neighborhood_preservation": safe_mean([r.get("neighborhood_preservation") for r in good]),
        "oe_damage":                 safe_mean([r.get("oe_damage") for r in good]),
        "locality_drop":             None,
    },
    "samples": results_memit,
}

out_path = "/content/memit_baseline_gpt2xl.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("  MEMIT BASELINE — GPT-2-XL — FINAL")
print("=" * 55)
m = summary["metrics"]
print(f"  Edit success:   {m['edit_success']:.1%}")
print(f"  Para success:   {m['paraphrase_success']:.1%}" if m['paraphrase_success'] else "  Para: no data")
print(f"  OE Bleed:       {m['neighborhood_bleed']:.1%}" if m['neighborhood_bleed'] else "  Bleed: no data")
print(f"  Nbr preserved:  {m['neighborhood_preservation']:.1%}" if m['neighborhood_preservation'] else "  Preservation: no data")
print("=" * 55)

from google.colab import files
files.download(out_path)

  MEMIT BASELINE — GPT-2-XL — FINAL
  Edit success:   92.6%
  Para success:   51.2%
  OE Bleed:       2.5%
  Nbr preserved:  8.1%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>